# The newsvendor problem

You buy stock **today** to sell **tomorrow**, and tomorrow's demand is unknown. Order too little and
you leave money on the table; order too much and you scrap the surplus at a loss.

This notebook builds four different answers to that one situation and compares them:

| | the question it answers |
|---|---|
| **Stochastic** | one order, chosen knowing only the *distribution* of demand |
| **Expected value** | one order, chosen as if demand were certain at its average |
| **Perfect information** | a different order per scenario — a vendor who can see tomorrow |
| **Max worst case** | the order that makes the worst outcome least bad |

The interesting content is the **gaps between them**. The gap to perfect information is what
clairvoyance would be worth; the gap to the expected-value answer is what modelling the uncertainty
is worth.

Everything below is built one step at a time. The streamlined version — the package that does all of
this in four calls — is at the bottom, after you have built it by hand.

## Licence setup — read this before running

This notebook needs a Gurobi licence. **It names three secrets and contains none of them**, which is
what makes it safe to share, copy, or hand to a student.

1. **In Colab** — add `GRB_WLSACCESSID`, `GRB_WLSSECRET` and `GRB_LICENSEID` once via the **key icon**
   in the left sidebar, then enable *this notebook* in the Secrets panel. The secret is bound to your
   Google account, not to the notebook, so everyone runs against their own licence.
2. **On a machine with a licence file** — `gp.Env()` finds `gurobi.lic` on its own.

If neither is available the cell stops with an instruction rather than a stack trace.

In [1]:
import gurobipy as gp

env = None
try:
    from google.colab import userdata
    try:
        params = {
            "WLSACCESSID": userdata.get("GRB_WLSACCESSID"),
            "WLSSECRET":   userdata.get("GRB_WLSSECRET"),
            "LICENSEID":   int(userdata.get("GRB_LICENSEID")),   # userdata returns a string
        }
    except userdata.SecretNotFoundError:
        raise SystemExit(
            "Add GRB_WLSACCESSID, GRB_WLSSECRET and GRB_LICENSEID as Colab Secrets "
            "(key icon, left sidebar), then re-run this cell."
        )
    except userdata.NotebookAccessError:
        raise SystemExit(
            "Those secrets exist but this notebook has not been granted access. "
            "Enable it in the Secrets panel, then re-run."
        )
    env = gp.Env(params=params)
    print("licence: Colab Secrets (WLS)")
except ImportError:
    env = gp.Env()
    print("licence: local gurobi.lic")

Set parameter Username


Set parameter LicenseID to value <removed>


Academic license - for non-commercial use only - expires 2026-12-04


licence: local gurobi.lic


## The economics

Four numbers describe the whole business. They are written out here rather than loaded from a config
file **on purpose** — each one carries a concept the narration explains, and seeing the number beside
the sentence is the lesson.

**`recover` is the one to look at.** It is the scrap price, and it is *negative*: you pay 3 per unit
to dispose of surplus. If `recover` were ever greater than or equal to `cost`, buying unlimited stock
purely to scrap it would be profitable and the model would have no optimum at all.

In [2]:
cost    = 2      # what you pay per unit
retail  = 15     # what you sell it for
recover = -3     # scrap "price" - NEGATIVE, you pay to dispose of surplus

print(f"margin on a sold unit   : {retail - cost:>6.2f}")
print(f"loss on a scrapped unit : {cost - recover:>6.2f}")

margin on a sold unit   :  13.00
loss on a scrapped unit :   5.00


Those two numbers are the whole tension. Selling one more unit earns 13; scrapping one more costs 5.
Because the reward for selling exceeds the penalty for scrapping, you should expect the best order to
sit **above** average demand — but by how much is exactly what the model is for.

## The demand scenarios

Demand is uncertain, so we approximate the distribution with a **sample** of 1,000 possible
tomorrows. More scenarios means a more accurate answer and a harder problem.

Unlike the four numbers above, these 1,000 values are **not** written into the notebook. They are
*instance data* — many entries, indexed by scenario, named nowhere in the prose — so they live in one
file that both this notebook and the package read. If each typed its own copy, a failed agreement
check at the bottom could not tell a typo in the data from a bug in a constraint.

In [3]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join("..", "..", "src")))
from orteach.data import load_demand
from orteach import tolerance

demand = load_demand()
n = len(demand)

print(f"scenarios : {n}")
print(f"mean      : {sum(demand)/n:8.2f}")
print(f"min / max : {min(demand):8.2f} / {max(demand):8.2f}")

scenarios : 1000
mean      :   402.28
min / max :    33.35 /   752.33


Print the first few as a table so the **shape** of the data is explicit — one row per scenario, one
demand each — rather than implied by punctuation.

In [4]:
for i in range(5):
    print(f"scenario {i:>4} -> demand {demand[i]:8.2f}")
print("...")

scenario    0 -> demand   291.59
scenario    1 -> demand   483.14
scenario    2 -> demand   314.90
scenario    3 -> demand   516.65
scenario    4 -> demand   403.83
...


## Before you go further — write down a number

Average demand is about 400. The margin on a sale is 13 and the loss on a scrap is 5.

**Write down the order quantity you think is best.** Above 400, below it, or exactly 400? You will
compare your number against four different answers, and a result you had no expectation about
teaches nothing, because there is nothing for it to contradict.

## First, score one order by hand

Before building any model, settle what "profit" even means. Pick an order of 400 — average demand,
the obvious guess — and work out what happens in a single scenario.

In [5]:
trial_order = 400
d0 = demand[0]

sold      = min(trial_order, d0)
scrapped  = trial_order - sold
profit_0  = -trial_order * cost + sold * retail + recover * scrapped

print(f"scenario 0 demand : {d0:8.2f}")
print(f"  units sold      : {sold:8.2f}")
print(f"  units scrapped  : {scrapped:8.2f}")
print(f"  profit          : {profit_0:8.2f}")

scenario 0 demand :   291.59
  units sold      :   291.59
  units scrapped  :   108.41
  profit          :  3248.68


Now the same arithmetic across all 1,000 scenarios. This is a loop over scenarios rather than a loop
hiding a decision — every pass does the identical thing to a different number, which is the kind of
repetition it is fine to fold up.

In [6]:
profits_400 = []
for d in demand:
    sold     = min(trial_order, d)
    scrapped = trial_order - sold
    profits_400.append(-trial_order * cost + sold * retail + recover * scrapped)

expected_400 = sum(profits_400) / n
print(f"order 400 -> expected profit {expected_400:10.2f}")
print(f"             worst scenario  {min(profits_400):10.2f}")

order 400 -> expected profit    4528.50
             worst scenario    -1399.67


## The stochastic model, built one piece at a time

Ordering 400 was a guess. The model chooses the order for us.

The structure that matters: **`order` is decided once, before demand is known.** `sales` and
`discount` are decided *per scenario*, after demand is revealed. That split — one decision now, a
recovery decision later — is what makes this a two-stage recourse problem rather than 1,000 separate
puzzles.

In [7]:
m = gp.Model(env=env)
tolerance.apply(m)          # solve as tightly as the assertion at the end claims
m.ModelSense = gp.GRB.MAXIMIZE
print(f"empty model created, sense = MAXIMIZE")

empty model created, sense = MAXIMIZE


Bounds on profit first. They are not required for correctness, but a variable free between plus and
minus infinity makes the solver work harder than it needs to.

Best case: every unit sells at retail. Worst case: buy the largest demand ever seen, scrap all of it,
and sell only the smallest.

In [8]:
hi = max(demand) * (retail - cost)
lo = max(demand) * (recover - cost) + min(demand) * retail
print(f"profit bounds: {lo:10.2f} .. {hi:10.2f}")

profit bounds:   -3261.38 ..    9780.30


Now the variables. `order` is a single number — **one** decision for all scenarios. The other three
are indexed by scenario, so there are 1,000 of each.

Note where the objective lives: `obj=1.0/n` on `profit` makes the objective the *average* profit
across scenarios, which is what "expected profit" means here.

In [9]:
order    = m.addVar(name="order")
profit   = m.addVars(n, obj=1.0/n, lb=lo, ub=hi, name="profit")
sales    = m.addVars(n, ub=demand, name="sales")
discount = m.addVars(n, name="discount")
m.update()

print(f"variables: {m.NumVars}  (1 order + 3 x {n} scenario variables)")

variables: 3001  (1 order + 3 x 1000 scenario variables)


`ub=demand` on `sales` is the line that encodes the whole uncertainty. It says: in scenario *i*, you
cannot sell more than that scenario's demand. Everything else follows from it.

Two constraint families. The first defines profit; the second says every unit ordered is either sold
or scrapped — nothing evaporates.

In [10]:
m.addConstrs((profit[i] == -order * cost + sales[i] * retail + recover * discount[i]
              for i in range(n)), name="profit")
m.addConstrs((sales[i] + discount[i] == order for i in range(n)), name="demand")
m.update()

print(f"constraints: {m.NumConstrs}")
assert m.NumConstrs > 0, "model built empty - check the API call order"

constraints: 2000


**Predict before you solve.** You wrote a number down earlier. The hand-built order of 400 earned
about 4,538 on average. Will the optimal order be higher or lower than 400, and will it beat 4,538 by
a little or a lot?

In [11]:
m.optimize()
sp_order   = order.X
sp_profits = [profit[i].X for i in range(n)]
sp_expected = sum(sp_profits) / n

print(f"optimal order   : {sp_order:10.2f}")
print(f"expected profit : {sp_expected:10.2f}")
print(f"worst scenario  : {min(sp_profits):10.2f}")

optimal order   :     458.25
expected profit :    4647.37
worst scenario  :   -1690.91


The order came out **above** average demand, as the margin arithmetic suggested it should: a lost
sale costs 13 and a scrapped unit costs 5, so it pays to over-order and eat some scrap.

Compare against the hand-built guess of 400 to see what the model actually bought you.

In [12]:
print(f"order 400 (guess)     : {expected_400:10.2f}")
print(f"order {sp_order:.2f} (optimal) : {sp_expected:10.2f}")
print(f"improvement           : {sp_expected - expected_400:10.2f}")
m.dispose()

order 400 (guess)     :    4528.50
order 458.25 (optimal) :    4647.37
improvement           :     118.88


## The expected-value trap

Here is the shortcut almost everyone reaches for first: *demand averages 400, so order 400 and be
done.* Replace the distribution with its mean and solve a problem with no uncertainty in it at all.

With demand certain, the answer is trivially "order exactly demand" — every unit sells, nothing
scraps. But the world still has 1,000 scenarios in it, so that order has to be scored against all of
them, honestly, using the same accounting as before.

**Predict:** you already computed this one by accident. Which number above is the expected-value
answer?

In [13]:
ev_order = sum(demand) / n
ev_profits = []
for d in demand:
    sold     = min(ev_order, d)
    scrapped = ev_order - sold
    ev_profits.append(-ev_order * cost + sold * retail + recover * scrapped)
ev_expected = sum(ev_profits) / n

print(f"EV order        : {ev_order:10.2f}")
print(f"expected profit : {ev_expected:10.2f}")

EV order        :     402.28
expected profit :    4537.92


The **Value of the Stochastic Solution** is the gap: what you gain by modelling the uncertainty
instead of averaging it away.

In [14]:
vss_value = sp_expected - ev_expected
print(f"stochastic  : {sp_expected:10.2f}")
print(f"expected value : {ev_expected:10.2f}")
print(f"VSS         : {vss_value:10.2f}   ({100*vss_value/ev_expected:.1f}% better)")
assert vss_value >= 0, "VSS is negative - the stochastic solution cannot be worse than EV"

stochastic  :    4647.37
expected value :    4537.92
VSS         :     109.46   (2.4% better)


## Perfect information

Now the other direction. Suppose you could see tomorrow's demand before ordering — a different,
perfectly chosen order in every scenario.

This is not achievable. It is an **upper bound**, and its distance from the stochastic answer prices
the uncertainty itself.

**Predict:** knowing demand exactly, what does the vendor order in each scenario, and how much does
it scrap?

In [15]:
pi_profits = [d * (retail - cost) for d in demand]
pi_expected = sum(pi_profits) / n

print(f"expected profit under perfect information : {pi_expected:10.2f}")
print(f"worst scenario                            : {min(pi_profits):10.2f}")

expected profit under perfect information :    5229.70
worst scenario                            :     433.57


Nothing is ever scrapped, so the scrap price never appears — knowing demand, you order exactly
demand. That is why the expression is simply demand times margin.

The **Expected Value of Perfect Information** is what a perfect forecast would be worth. It is the
most any forecasting effort could possibly earn you.

In [16]:
evpi_value = pi_expected - sp_expected
print(f"perfect information : {pi_expected:10.2f}")
print(f"stochastic          : {sp_expected:10.2f}")
print(f"EVPI                : {evpi_value:10.2f}")
assert pi_expected >= sp_expected >= ev_expected, "the chain of bounds is out of order"
print("\nbounds ordered correctly: PI >= SP >= EV")

perfect information :    5229.70
stochastic          :    4647.37
EVPI                :     582.33

bounds ordered correctly: PI >= SP >= EV


## Changing the question: risk instead of average

Everything so far maximised the *average*. A vendor who cannot survive a bad night does not care
about the average.

So change the objective: maximise the **worst** scenario's profit. One new variable, `worst`, pushed
up underneath every scenario's profit.

**Predict — this is the one worth guessing.** What order quantity maximises the worst case, and what
does that do to expected profit?

In [17]:
m = gp.Model(env=env)
tolerance.apply(m)          # solve as tightly as the assertion at the end claims
m.ModelSense = gp.GRB.MAXIMIZE

worst    = m.addVar(lb=lo, ub=hi, obj=1, name="worst")
order    = m.addVar(name="order")
profit   = m.addVars(n, lb=lo, ub=hi, name="profit")
sales    = m.addVars(n, ub=demand, name="sales")
discount = m.addVars(n, name="discount")

m.addConstrs((profit[i] == -order * cost + sales[i] * retail + recover * discount[i]
              for i in range(n)), name="profit")
m.addConstrs((sales[i] + discount[i] == order for i in range(n)), name="demand")
m.addConstrs((worst <= profit[i] for i in range(n)), name="worst")

m.optimize()
rn_order = sp_order
w_order   = order.X
w_profits = [profit[i].X for i in range(n)]
w_expected = sum(w_profits) / n

print(f"max-worst-case order : {w_order:10.2f}")
print(f"expected profit      : {w_expected:10.2f}")
print(f"worst scenario       : {min(w_profits):10.2f}")
m.dispose()

max-worst-case order :      33.35
expected profit      :     433.57
worst scenario       :     433.57


Look at what it chose. The order collapsed to roughly the **smallest demand in the sample** — order
only what you are certain to sell, and you can never be caught with surplus.

The floor is now the best floor available. The average is a fraction of what it was.

In [18]:
print(f"{'':22} {'order':>10} {'E[profit]':>12} {'worst':>12}")
print(f"{'risk neutral':22} {rn_order:10.2f} {sp_expected:12.2f} {min(sp_profits):12.2f}")
print(f"{'max worst case':22} {w_order:10.2f} {w_expected:12.2f} {min(w_profits):12.2f}")
print()
print(f"floor improved by : {min(w_profits) - min(sp_profits):10.2f}")
print(f"average gave up   : {sp_expected - w_expected:10.2f}")

                            order    E[profit]        worst
risk neutral               458.25      4647.37     -1690.91
max worst case              33.35       433.57       433.57

floor improved by :    2124.48
average gave up   :    4213.80


That trade is the actual content of risk-averse optimisation. Safety is purchasable and it is not
cheap, and the model makes the exchange rate explicit instead of leaving it to temperament.

## All four together

In [19]:
rows = [
    ("expected value",      ev_order, ev_expected, min(ev_profits)),
    ("stochastic",          sp_order, sp_expected, min(sp_profits)),
    ("perfect information", float("nan"), pi_expected, min(pi_profits)),
    ("max worst case",      w_order,  w_expected,  min(w_profits)),
]
print(f"{'':22} {'order':>12} {'E[profit]':>12} {'worst':>12}")
for name, o, e, w in rows:
    shown = "per-scenario" if o != o else f"{o:12.2f}"
    print(f"{name:22} {shown:>12} {e:12.2f} {w:12.2f}")

print()
print(f"VSS  (modelling uncertainty is worth) : {vss_value:10.2f}")
print(f"EVPI (a perfect forecast is worth)    : {evpi_value:10.2f}")

                              order    E[profit]        worst
expected value               402.28      4537.92     -1411.10
stochastic                   458.25      4647.37     -1690.91
perfect information    per-scenario      5229.70       433.57
max worst case                33.35       433.57       433.57

VSS  (modelling uncertainty is worth) :     109.46
EVPI (a perfect forecast is worth)    :     582.33


---

# Now the streamlined version

You have now built every one of these by hand, so the abstraction below hides nothing you have not
already seen. It is worth wrapping at this point because the next thing anyone does with this model
is run it a dozen times at different prices, and repeating forty cells to change one number is not a
lesson, it is typing.

`orteach.newsvendor` holds the same four models as functions. It takes the demand data **as an
argument** and never re-reads the file, so if you edit a demand value above, it flows into both the
hand-built model and the check below.

In [20]:
from orteach import newsvendor as nv
from orteach.tolerance import AGREEMENT_RTOL, rel_diff

packaged_sp = nv.solve_stochastic(demand, cost, retail, recover, env=env)
packaged_ev = nv.solve_expected_value(demand, cost, retail, recover)
packaged_pi = nv.solve_perfect_information(demand, cost, retail, recover)
packaged_w  = nv.solve_max_worst_case(demand, cost, retail, recover, env=env)

print(f"{'':22} {'order':>12} {'E[profit]':>12}")
for r in (packaged_ev, packaged_sp, packaged_pi, packaged_w):
    shown = "per-scenario" if r.order != r.order else f"{r.order:12.2f}"
    print(f"{r.label:22} {shown:>12} {r.expected_profit:12.2f}")

                              order    E[profit]
expected value               402.28      4537.92
stochastic                   458.25      4647.37
perfect information    per-scenario      5229.70
max worst case                33.35       433.57


## The agreement assertion

The package and this notebook now hold the same model twice, deliberately. That is fine — the
notebook builds it by hand because that is the lesson, the package builds it once because that is the
code — but deliberate duplication with nothing comparing the copies is just duplication with a story
attached.

So compare them. Every number the notebook produced by hand, against the same number from the
package.

One more thing the check relies on. Both sides solved at **tighter-than-default** solver
tolerances — `orteach.tolerance` sets Gurobi to `1e-9` optimality and feasibility, and a zero
MIP gap. Without that, a notebook asserting `1e-9` agreement against a solver that only promised
`1e-6` is not testing that the two models agree; it is testing that they took the same path on the
same machine, and it fails the first time it runs somewhere else. The tolerance itself is a single
constant in the package, imported here, not typed again.


In [21]:
checks = [
    ("stochastic order",    sp_order,     packaged_sp.order),
    ("stochastic profit",   sp_expected,  packaged_sp.expected_profit),
    ("expected-value profit", ev_expected, packaged_ev.expected_profit),
    ("perfect-info profit", pi_expected,  packaged_pi.expected_profit),
    ("max-worst-case order", w_order,     packaged_w.order),
    ("max-worst-case profit", w_expected, packaged_w.expected_profit),
]

worst_rel = 0.0
for name, hand, packaged in checks:
    rel = rel_diff(hand, packaged)
    worst_rel = max(worst_rel, rel)
    print(f"{name:24} hand {hand:12.4f}   package {packaged:12.4f}   rel {rel:.2e}")

assert worst_rel < AGREEMENT_RTOL, f"notebook and package disagree by {worst_rel:.2e}"
print(f"\nnotebook and package agree to {worst_rel:.1e}")

stochastic order         hand     458.2478   package     458.2478   rel 0.00e+00
stochastic profit        hand    4647.3748   package    4647.3748   rel 0.00e+00
expected-value profit    hand    4537.9162   package    4537.9162   rel 0.00e+00
perfect-info profit      hand    5229.7016   package    5229.7016   rel 0.00e+00
max-worst-case order     hand      33.3516   package      33.3516   rel 0.00e+00
max-worst-case profit    hand     433.5705   package     433.5705   rel 0.00e+00

notebook and package agree to 0.0e+00


---

## Where to take this next

- Change `recover` to `0` — free disposal — and re-run. Which of the four answers moves most, and why?
- Push `sigma` up in `orteach.data` and regenerate the table. Does EVPI grow or shrink?
- Maximising the worst case is the most extreme risk stance available. **CVaR** — the average of the
  worst 25% rather than the single worst — sits between it and the risk-neutral answer, and needs one
  more variable per scenario.